# PostgreSQL setup validation

This notebook validates the adapted SQL deployment in PostgreSQL.

It checks:
- DB connectivity
- target schema existence
- table/function counts
- key object presence
- basic function smoke checks

In [1]:
import os
from pathlib import Path

import psycopg
from psycopg.rows import dict_row


def load_env(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env(Path.cwd().parent / ".env")

DB_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE"),
    "user": os.getenv("PGUSER"),
    "password": os.getenv("PGPASSWORD"),
}
SCHEMA = "s_grnplm_as_t_didsd_nnn_db_tmd"

conn = psycopg.connect(**DB_CONFIG, row_factory=dict_row)
print("Connected")

Connected


In [2]:
with conn.cursor() as cur:
    cur.execute("SELECT current_database() AS db, current_user AS usr, version() AS ver")
    row = cur.fetchone()
row

{'db': 'giga4dqm',
 'usr': 'giga4dqm',
 'ver': 'PostgreSQL 16.13 (Debian 16.13-1.pgdg13+1) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit'}

In [3]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT EXISTS (
            SELECT 1
            FROM information_schema.schemata
            WHERE schema_name = %s
        ) AS schema_exists
        """,
        (SCHEMA,),
    )
    schema_exists = cur.fetchone()["schema_exists"]

print({"schema": SCHEMA, "exists": schema_exists})

{'schema': 's_grnplm_as_t_didsd_nnn_db_tmd', 'exists': True}


In [4]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            (SELECT count(*)
             FROM information_schema.tables
             WHERE table_schema = %s) AS table_count,
            (SELECT count(*)
             FROM pg_proc p
             JOIN pg_namespace n ON n.oid = p.pronamespace
             WHERE n.nspname = %s) AS function_count
        """,
        (SCHEMA, SCHEMA),
    )
    counts = cur.fetchone()

counts

{'table_count': 17, 'function_count': 38}

In [5]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = %s
        ORDER BY table_name
        """,
        (SCHEMA,),
    )
    tables = [r["table_name"] for r in cur.fetchall()]

tables

['d_agr_cred',
 'd_agr_cred_coa',
 'd_agr_cred_coa_period_calc_dt',
 'd_agr_cred_coa_period_prep_bal',
 'd_agr_cred_coa_period_prep_main_debts',
 'd_agr_cred_optn',
 'd_agr_cred_prvsn_period',
 'd_settings',
 'd_tech_tbl_coa_bal_h',
 'debug_log',
 'etl_bkmart_log',
 'etl_task_param',
 't_coa',
 't_je_line',
 't_spt_je_header_fltr',
 't_src_system_type',
 't_str_pilot_tune']

In [6]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT proname
        FROM pg_proc p
        JOIN pg_namespace n ON n.oid = p.pronamespace
        WHERE n.nspname = %s
        ORDER BY proname
        """,
        (SCHEMA,),
    )
    funcs = [r["proname"] for r in cur.fetchall()]

funcs[:50]

['add_arr_log',
 'fn_create_obj_test1',
 'fn_dbc_partition_table_name_for',
 'fn_drop_gen_data_tables',
 'fn_explain_to_debug',
 'fn_gen_data_test1',
 'fn_gen_data_test2',
 'fn_gen_data_test3',
 'fn_generate_coa_data',
 'fn_generate_je_line_data',
 'fn_run_calc_d_agr_cred_coa_period_calc_dt_test3',
 'fn_run_calc_fltr_skew_test1',
 'fn_run_calc_fltr_test1',
 'fn_run_calc_main_debts_test2',
 'fn_run_calc_skew_test1',
 'fn_run_test1',
 'fn_run_test2',
 'fn_run_test3',
 'gen_random_uuid',
 'generate_agr_cred',
 'generate_agr_cred_coa',
 'generate_agr_cred_coa_period_prep_bal',
 'generate_agr_cred_optn',
 'generate_tech_tbl_coa_bal_h',
 'prc_debug',
 'prc_error_message',
 'save_prc_log',
 'sp_collect_stat',
 'sp_dm_ent_ld_d_agr_cred_coa_period_calc_dt',
 'sp_dm_ent_ld_d_agr_cred_coa_period_calc_dt_separate',
 'sp_dm_ent_ld_d_agr_cred_coa_period_prep_main_debts',
 'sp_dm_ent_ld_d_agr_cred_coa_period_prep_main_debts_separate',
 'sp_execute_run',
 'sp_get_calc_params',
 'sp_spt_je_header_fltr'

In [7]:
# Smoke-check: helper fallback function
with conn.cursor() as cur:
    cur.execute(
        "SELECT s_grnplm_as_t_didsd_nnn_db_tmd.fn_dbc_partition_table_name_for(%s, %s, %s) AS result",
        (SCHEMA, "t_je_line", "000004"),
    )
    helper_result = cur.fetchone()["result"]

helper_result

's_grnplm_as_t_didsd_nnn_db_tmd.t_je_line'

In [8]:
# Smoke-check: main object-create function now exists and is callable
# (safe no-op in PostgreSQL-adapted version)
with conn.cursor() as cur:
    cur.execute("SELECT s_grnplm_as_t_didsd_nnn_db_tmd.fn_create_obj_test1()")

print("fn_create_obj_test1() executed")

fn_create_obj_test1() executed


In [9]:
conn.close()
print("Connection closed")

Connection closed
